# 🍗 Predicción de Demanda Operativa (Preparación Proactiva)

## Contexto de Negocio
La operación de cocina en un QSR (Quick Service Restaurant) depende de la capacidad de anticipar la demanda diaria.

## Objetivo Profesional
Implementar un pipeline de forecasting probabilístico que entregue **bandas de decisión operativa** (P10, P50, P90) y evalúe el impacto financiero.

---

## 1. 📦 Setup y Conexión

In [21]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector
from dotenv import load_dotenv
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (16, 7)
plt.rcParams['font.size'] = 12

load_dotenv()

# Configurar Tracking URI central en la raíz del proyecto para que dbt y el back lo encuentren
current_path = os.getcwd()
root_path = os.path.abspath(os.path.join(current_path, "..", "..", ".."))
mlruns_path = f"file:///{root_path.replace('\\', '/')}/mlruns"
mlflow.set_tracking_uri(mlruns_path)

print(f'✅ Tracking URI configurado en: {mlruns_path}')
print('✅ Entorno profesional configurado')

✅ Tracking URI configurado en: file:///c:/Users/Nick/Documents/Dat372 - BI/HGC/hgc-system/mlruns
✅ Entorno profesional configurado


In [22]:
# Conexión a Snowflake
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema='TRAINING_DATASETS'
)

query = 'SELECT * FROM TRAINING_DATASETS.OBT_OPERATIONAL_DEMAND ORDER BY FECHA_PREDICCION'
df = pd.read_sql(query, conn)
df.columns = [c.upper() for c in df.columns]
df['FECHA_PREDICCION'] = pd.to_datetime(df['FECHA_PREDICCION'])

print(f'✅ Dataset cargado: {df.shape[0]:,} observaciones diarias')
print(f'🍗 Productos: {df["NOMBRE_PRODUCTO"].nunique()}')

✅ Dataset cargado: 38,241 observaciones diarias
🍗 Productos: 9


## 2. 🔍 EDA y Feature Engineering

In [23]:
# Selección del producto core
target_product = df.groupby('NOMBRE_PRODUCTO')['TARGET_CANTIDAD_TOTAL'].sum().idxmax()
df_p = df[df['NOMBRE_PRODUCTO'] == target_product].copy()
df_p = df_p.set_index('FECHA_PREDICCION').sort_index()

def add_cyclic_features(df):
    df['DOW_SIN'] = np.sin(2 * np.pi * df['FEATURE_DIA_SEMANA'] / 7)
    df['DOW_COS'] = np.cos(2 * np.pi * df['FEATURE_DIA_SEMANA'] / 7)
    df['MONTH_SIN'] = np.sin(2 * np.pi * df['FEATURE_MES'] / 12)
    df['MONTH_COS'] = np.cos(2 * np.pi * df['FEATURE_MES'] / 12)
    return df

df_p = add_cyclic_features(df_p)
features = [
    'DOW_SIN', 'DOW_COS', 'MONTH_SIN', 'MONTH_COS', 
    'FEATURE_ES_FIN_DE_SEMANA', 'FEATURE_LAG_1_DIA', 'FEATURE_LAG_7_DIAS',
    'FEATURE_ROLLING_AVG_7D', 'FEATURE_ROLLING_STD_7D', 'FEATURE_ROLLING_AVG_30D'
]
print(f'✅ Features generados: {len(features)}')

✅ Features generados: 10


## 3. 🔥 Entrenamiento y Registro en MLflow

In [24]:
mlflow.set_experiment('HGC_Operational_Demand')

quantiles = [0.05, 0.5, 0.95]
horizon = 60
train_df = df_p.iloc[:-horizon]
test_df = df_p.iloc[-horizon:]

with mlflow.start_run(run_name='Operational_Demand_Full_Pipeline') as run:
    mlflow.log_params({
        'product': target_product,
        'algo': 'LGBM Quantile',
        'features': str(features)
    })
    
    for q in quantiles:
        model = lgb.LGBMRegressor(
            objective='quantile', 
            alpha=q, 
            n_estimators=300, 
            learning_rate=0.03, 
            num_leaves=24,
            verbosity=-1
        )
        model.fit(train_df[features], train_df['TARGET_CANTIDAD_TOTAL'])
        
        test_df[f'P{int(q*100)}'] = model.predict(test_df[features])
        
        # Loguear solo el modelo de el cuantil 0.5 como el principal para serving del back
        if q == 0.5:
            # Registro del modelo en el registry
            mlflow.lightgbm.log_model(
                lgb_model=model, 
                artifact_path='model',
                registered_model_name='OperationalDemandModel'
            )
    
    # Metricas (P50)
    wape = np.sum(np.abs(test_df['TARGET_CANTIDAD_TOTAL'] - test_df['P50'])) / np.sum(test_df['TARGET_CANTIDAD_TOTAL'])
    mlflow.log_metric('wape', wape)
    
    print(f'✅ Modelo entrenado y registrado como "OperationalDemandModel" en MLflow')
    print(f'🎯 WAPE: {wape*100:.2f}%')

2026/04/16 07:56:36 INFO mlflow.tracking.fluent: Experiment with name 'HGC_Operational_Demand' does not exist. Creating a new experiment.
2026/04/16 07:56:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 07:56:36 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Modelo entrenado y registrado como "OperationalDemandModel" en MLflow
🎯 WAPE: 14.26%


Successfully registered model 'OperationalDemandModel'.
Created version '1' of model 'OperationalDemandModel'.


## 4. 🚀 Instrucciones para Despliegue (Serving)

Una vez registrado el modelo, puedes levantarlo como un servidor API ejecutando el siguiente comando en tu terminal (en la raíz del proyecto):

```bash
mlflow models serve -m "models:/OperationalDemandModel/latest" -p 5001 --no-conda
```

Esto expondrá un endpoint en `http://127.0.0.1:5001/invocations` que el backend usará para las predicciones interactivas.

In [25]:
conn.close()
print('🔌 Conexión cerrada.')

🔌 Conexión cerrada.
